# Coffee Roastery Knowledge Graph

A minimal end-to-end knowledge graph in **Cognite Data Fusion (CDF)** built on **Data Modeling (DM)** and the **Core Data Model (CDM)**.

## What is a knowledge graph in CDF?

A graph is just **typed nodes connected by typed edges**. In CDF the types come from the CDM — `CogniteAsset`, `CogniteTimeSeries`, `CogniteActivity`, `CogniteFile`, and so on — and connections take two forms:

- **Direct relations** — 1:1 / 1:N pointers stored as a property on the node, e.g. `time_series.assets`, `asset.parent`. Cheap to read, fixed at write time.
- **Edges** — separate, addressable instances with their own properties. Used here for `CogniteDiagramAnnotation`, where each link carries its own bounding box and confidence score.

Why a graph rather than a star schema? Because in an industrial plant you traverse: *"For this work order, what asset? What sensors? What did the temperature do during the work? Which P&ID shows it?"* That walk crosses four different data types — exactly what the graph models.

## What we build

```
Coffee Roastery (root)
 ├── Bean Hopper          ── tagged BH-01 in P&ID
 ├── Roasting Drum        ── tagged RD-01
 │    ├── [TS] Drum Temperature   ── tagged TI-101
 │    └── [TS] Bean Moisture      ── tagged MI-101
 └── Cooling Tray         ── tagged CT-01

Files
 ├── P&ID (PDF)              → Coffee Roastery
 └── Maintenance Manual      → Roasting Drum

Activities
 ├── Monthly Drum Cleaning   → Roasting Drum   (scheduled — leaves a flat patch in the data)
 └── Smoke Alarm event       → Roasting Drum   (issue — correlates with a temp spike)

Annotation edges (P&ID → Assets / Time Series)
 BH-01 → Bean Hopper      RD-01 → Roasting Drum    CT-01 → Cooling Tray
 TI-101 → Drum Temperature                          MI-101 → Bean Moisture
```

Every reference uses `instance_id` (space + external_id) — no classic-API external IDs.

## How this maps to a real deployment

The notebook fakes ingestion by writing nodes from the SDK. In a real deployment each step is owned by a different system and pushed in by one of the [Cognite extractors](https://www.cognite.com/en/product/extractors) (SAP, DB, PI, PI AF, OPC UA, OPC Classic, MQTT, file, Documentum, …) or by a CDF Transformation / Function. Each step calls out which one does the equivalent job in production.

After every step, open [Fusion](https://fusion.cognite.com) and verify the data shows up. Each step ends with a short **Verify in Fusion** section listing exactly what to look for.

## Setup

A **space** is a namespace for your DM instances — like a database schema. All coffee-roastery data in this tutorial lives in one space (`coffee_roastery_demo`). CDM's own type definitions live in the system space `cdf_cdm`.

In [ ]:
import sys
import tempfile
import time
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np

from cognite.client.data_classes import filters
from cognite.client.data_classes.data_modeling import (
    DirectRelationReference,
    EdgeApply,
    NodeApply,
    NodeId,
    NodeOrEdgeData,
    SpaceApply,
    ViewId,
)
from cognite.client.data_classes.data_modeling.cdm.v1 import (
    CogniteActivityApply,
    CogniteAssetApply,
    CogniteFileApply,
    CogniteTimeSeriesApply,
)

sys.path.insert(0, str(Path.cwd().parent))
from helpers import CDM_SPACE, CDM_UNITS_SPACE, CDM_VERSION, SPACE, get_client

In [ ]:
client = get_client(client_name="coffee-roastery-demo")

In [ ]:
client.data_modeling.spaces.apply(SpaceApply(space=SPACE, name="Coffee Roastery Demo"))
print(f"Space '{SPACE}' ready")

## Step 1 — Asset Hierarchy

`CogniteAsset` represents a **functional location** — a system, sub-system, or piece of equipment in an industrial process. The hierarchy is built from a single direct relation, `parent`. CDF derives `root`, `path`, and `children` automatically; we only ever write `parent`.

The asset hierarchy is **master data** owned by an ERP / EAM system, where the functional-location structure is maintained by the engineering team. CDF mirrors it with one of the [Cognite extractors](https://www.cognite.com/en/product/extractors):

| Source | Extractor |
|---|---|
| SAP ERP | **Cognite SAP Extractor** |
| Any ODBC-compatible database — Oracle, MySQL, Postgres — including the database that backs most CMMS systems | **Cognite DB Extractor** |
| CSV / Excel master-data exports staged in CDF RAW | **CDF Transformations** (SQL ETL inside CDF — not an extractor, a CDF feature) |

Real plants typically have **a few thousand to a few hundred thousand functional locations**. We use four nodes — one root and three children — so the demo fits on one diagram and one screen.

In [ ]:
roastery = CogniteAssetApply(
    space=SPACE,
    external_id="coffee_roastery",
    name="Coffee Roastery",
    description="Main facility — small-batch artisan coffee roaster",
)

hopper = CogniteAssetApply(
    space=SPACE,
    external_id="bean_hopper",
    name="Bean Hopper",
    description="Green-bean storage and feed hopper (25 kg capacity)",
    parent=DirectRelationReference(SPACE, "coffee_roastery"),
)

drum = CogniteAssetApply(
    space=SPACE,
    external_id="roasting_drum",
    name="Roasting Drum",
    description="Gas-fired rotating drum roaster (15 kg batch)",
    parent=DirectRelationReference(SPACE, "coffee_roastery"),
)

tray = CogniteAssetApply(
    space=SPACE,
    external_id="cooling_tray",
    name="Cooling Tray",
    description="Air-cooled tray with stirring arms for rapid bean cooling",
    parent=DirectRelationReference(SPACE, "coffee_roastery"),
)

result = client.data_modeling.instances.apply(nodes=[roastery, hopper, drum, tray])
print(f"Created {len(result.nodes)} assets:")
for n in result.nodes:
    print(f"  {n.space}/{n.external_id}")

### ✅ Verify in Fusion

Open [Fusion](https://fusion.cognite.com) → **Search** → type `Coffee Roastery`. You should see:

- The root asset *Coffee Roastery*.
- Its three children *Bean Hopper*, *Roasting Drum*, *Cooling Tray* listed under **Children** on the detail page.
- The **Path** breadcrumb on each child showing `Coffee Roastery / Bean Hopper` (or similar) — confirming CDF derived `path` from `parent`.

## Step 2 — Time Series

`CogniteTimeSeries` is the **metadata node** for a stream of timestamped values. It carries name, description, unit, and the step/numeric flag — but no datapoints. CDF stores datapoints in a separate, purpose-built time series engine; the metadata node is the handle the engine uses to find them.

Time series in industrial plants come almost exclusively from **process control and historian systems** — OSIsoft PI, the OPC UA layer exposed by modern PLCs and DCSes, OPC Classic on legacy systems, and MQTT brokers used by sensors and edge gateways. CDF connects to each with the matching [Cognite extractor](https://www.cognite.com/en/product/extractors):

| Source | Extractor |
|---|---|
| OSIsoft PI server | **Cognite PI Extractor** |
| OSIsoft PI Asset Framework | **Cognite PI AF Extractor** |
| OPC UA (PLCs, DCS, SCADA gateways) | **Cognite OPC UA Extractor** |
| OPC DA / OPC HDA (legacy OPC Classic) | **Cognite OPC Classic Extractor** |
| MQTT brokers (sensors, edge gateways) | **Cognite MQTT Extractor** |

The extractor auto-discovers tags from the source and creates one `CogniteTimeSeries` per tag. The asset link is **not** populated at this point — the historian doesn't usually know which functional location a tag belongs to. We will create the metadata first, then add the link.

We create two sensors on the roasting drum:

- **Drum Temperature** — thermocouple inside the drum (°C, with a CDM unit reference).
- **Bean Moisture** — NIR moisture sensor on the bean bed (%).

In [ ]:
ts_temp = CogniteTimeSeriesApply(
    space=SPACE,
    external_id="drum_temperature",
    name="Drum Temperature",
    description="Thermocouple inside the roasting drum",
    is_step=False,
    time_series_type="numeric",
    source_unit="°C",
    unit=DirectRelationReference(CDM_UNITS_SPACE, "temperature:deg_c"),
)

ts_moisture = CogniteTimeSeriesApply(
    space=SPACE,
    external_id="bean_moisture",
    name="Bean Moisture",
    description="NIR moisture sensor on the bean bed",
    is_step=False,
    time_series_type="numeric",
    source_unit="%",
    unit=DirectRelationReference(CDM_UNITS_SPACE, "dimensionless_ratio:percent"),
)

result = client.data_modeling.instances.apply(nodes=[ts_temp, ts_moisture])
print(f"Created {len(result.nodes)} time series (not yet linked to assets):")
for n in result.nodes:
    print(f"  {n.space}/{n.external_id}")

### ✅ Verify in Fusion

Search for `Drum Temperature` in Fusion. You should see:

- A time series result with the unit `°C` and description we set.
- The **Assets** section on the detail page is **empty** — the time series is not yet attached to any equipment in the graph. That mirrors the real-world state right after a historian extractor first connects.

## Step 3 — Contextualize: link Time Series → Assets (upsert)

A **view** is a typed lens over stored properties — `CogniteTimeSeries` at version `v1` is one view in the CDM. When we upsert, we name the view and supply only the fields we want to change.

The time series we just created arrived from a historian (PI, SCADA, OPC UA, …) that knows tag names but not which piece of equipment each tag belongs to. **Contextualization** is the process of resolving that mapping — making the graph traversable from a sensor to its asset and back.

> **Typed vs generic writes:** Steps 1 and 2 used the friendly typed classes (`CogniteAssetApply`, `CogniteTimeSeriesApply`). Here we switch to the generic `NodeApply` and name just the `assets` field — CDF merges it into the existing node. Edges (Step 7) use `EdgeApply` for the same reason: annotations are edges with their own properties and have no typed helper.

There are three common ways to do contextualization in CDF:

| Method | When to use |
|---|---|
| **CDF Transformations** (SQL on RAW staging) | The mapping is deterministic from the tag name — e.g. `RD-01.TI-101` → asset `roasting_drum`. The most common pattern when naming conventions are clean. |
| **Entity Matching** (text similarity + ML) | Names and descriptions are inconsistent across systems — e.g. SCADA tags `T101_CH1` mapping to *Roasting Drum*. |
| **Manual mapping in Fusion** | One-off corrections or small volumes maintained by an engineer. |

Whatever method picks the asset, the **write back to CDF always looks the same**: an **upsert** on the time series node, supplying only the `assets` direct relation. CDF merges the new property in, and the version counter increments — confirming we updated the existing node rather than creating a new one. This is the canonical CDM write pattern, and the rest of this notebook reuses it.

In [ ]:
ts_view = ViewId(CDM_SPACE, "CogniteTimeSeries", CDM_VERSION)
drum_asset = [DirectRelationReference(space=SPACE, external_id="roasting_drum")]

ts_temp_link = NodeApply(
    space=SPACE,
    external_id="drum_temperature",
    sources=[NodeOrEdgeData(source=ts_view, properties={"assets": drum_asset})],
)

ts_moisture_link = NodeApply(
    space=SPACE,
    external_id="bean_moisture",
    sources=[NodeOrEdgeData(source=ts_view, properties={"assets": drum_asset})],
)

result = client.data_modeling.instances.apply(nodes=[ts_temp_link, ts_moisture_link])
print(f"Upserted {len(result.nodes)} time series — now linked to 'roasting_drum':")
for n in result.nodes:
    print(f"  {n.space}/{n.external_id} (version {n.version})")

### ✅ Verify in Fusion

Open *Drum Temperature* in Fusion again (the same page as in Step 2). You should now see:

- The **Assets** section now lists *Roasting Drum* — the link we just wrote.
- Clicking *Roasting Drum* takes you to the asset, where *Drum Temperature* and *Bean Moisture* both appear under **Time series**. The graph is now traversable in both directions.
- The node's **version** has incremented (was 1, now 2) — confirming this was an upsert, not a re-create.

## Step 4 — Datapoints

Datapoints are the actual sensor values: a pair of `(timestamp, value)` for each reading. CDF stores them in a column-oriented engine that handles millions of points per second per project, and serves automatic aggregates (min/max/avg/count at multiple granularities) without precomputation. The metadata node we created in Step 2 is the address — datapoints are written and read against `instance_id`.

In a real plant, datapoints stream continuously from the same extractor that created the metadata node. The **Cognite PI Extractor** and **Cognite OPC UA Extractor** push values at the source scan rate — typically sub-second to a few seconds per tag. The **Cognite MQTT Extractor** handles event-driven sensors and edge gateways. The **Cognite OPC Classic Extractor** covers legacy OPC DA / HDA installations.

We synthesize **two weeks of 15-minute samples** with a daily roast cycle and **two embedded anomalies**. The anomalies are deliberate — they make the graph interesting later, when we want to correlate sensor data with maintenance events.

| Signal | Baseline pattern | Anomaly: ~10 days ago | Anomaly: ~5 days ago |
|---|---|---|---|
| Temperature | Daily cycle, ambient ~25 °C → roast peak ~220 °C | Drum off, flat at ~28 °C for 4 h | Chaotic spike to ~260 °C over ~1 h, then erratic cooldown and a 1 h dead zone at ambient |
| Moisture | Drops from ~11 % (green) to ~3 % (roasted) per batch | Stays high (~11.5 %) — no roasting | Crashes to ~1 % (scorched), then jumps back to ambient after shutdown |

In [ ]:
now = datetime.now(timezone.utc)
two_weeks_ago = now - timedelta(weeks=2)

# Activity timestamps (relative to now) — must match Step 5
maintenance_start = now - timedelta(days=10, hours=1)
maintenance_end = now - timedelta(days=10, hours=-3)  # 4-hour window
smoke_alarm_time = now - timedelta(days=5, hours=9, minutes=28)  # ~14:32 local

timestamps = []
current = two_weeks_ago
while current <= now:
    timestamps.append(int(current.timestamp() * 1000))
    current += timedelta(minutes=15)

n = len(timestamps)
np.random.seed(42)
hours_elapsed = np.arange(n) / 4.0

# ── Base patterns: daily roast cycles ─────────────────────────────
daytime = np.clip(np.sin(2 * np.pi * (hours_elapsed - 6) / 24), 0, 1)
temp_values = 25 + 195 * daytime + np.random.normal(0, 3, n)
moisture_values = 11 - 8 * daytime + np.random.normal(0, 0.3, n)

# ── Anomaly 1: Maintenance window (~10 days ago, 4 hours) ────────
#    Drum is off — temperature drops to ambient, moisture stays high
maint_start_ms = int(maintenance_start.timestamp() * 1000)
maint_end_ms = int(maintenance_end.timestamp() * 1000)
for i, ts in enumerate(timestamps):
    if maint_start_ms <= ts <= maint_end_ms:
        temp_values[i] = 28 + np.random.normal(0, 1)  # ambient ~28 °C
        moisture_values[i] = 11.5 + np.random.normal(0, 0.2)  # no roasting

# ── Anomaly 2: Smoke alarm (~5 days ago) ──────────────────────────
#    Chaff buildup blocks the exhaust over ~1 hour: temperature climbs
#    erratically past 260 °C, moisture crashes to near-zero (scorched
#    beans). Operator kills the drum at the alarm; temperature drops
#    chaotically, then a 1-hour dead zone while they clear the duct.
smoke_ms = int(smoke_alarm_time.timestamp() * 1000)
for i, ts in enumerate(timestamps):
    minutes_after = (ts - smoke_ms) / 60_000
    if -60 <= minutes_after <= 0:
        # Overheating buildup — exhaust increasingly blocked
        frac = (minutes_after + 60) / 60  # 0 → 1 over the hour
        temp_values[i] = 220 + 40 * frac**1.5 + np.random.normal(0, 5)
        moisture_values[i] = 3.0 - 2.0 * frac + np.random.normal(0, 0.2)
    elif 0 < minutes_after <= 30:
        # Alarm triggered — erratic cooldown as operator intervenes
        frac = minutes_after / 30
        temp_values[i] = 260 - 180 * frac + np.random.normal(0, 8)
        moisture_values[i] = 1.0 + 3.0 * frac + np.random.normal(0, 0.3)
    elif 30 < minutes_after <= 90:
        # Dead zone — drum is off while they clear the exhaust duct
        temp_values[i] = 30 + np.random.normal(0, 1.5)
        moisture_values[i] = 11.0 + np.random.normal(0, 0.3)

temp_values = np.clip(temp_values, 20, 270)
moisture_values = np.clip(moisture_values, 0.5, 12)

client.time_series.data.insert(
    instance_id=NodeId(SPACE, "drum_temperature"),
    datapoints=list(zip(timestamps, temp_values.tolist())),
)
client.time_series.data.insert(
    instance_id=NodeId(SPACE, "bean_moisture"),
    datapoints=list(zip(timestamps, moisture_values.tolist())),
)

print(f"Inserted {n} datapoints per time series")
print(f"Range: {two_weeks_ago:%Y-%m-%d %H:%M} → {now:%Y-%m-%d %H:%M} UTC")
print("\nEmbedded anomalies:")
print(
    f"  Maintenance window:  {maintenance_start:%Y-%m-%d %H:%M} → {maintenance_end:%Y-%m-%d %H:%M}"
)
print(f"  Smoke alarm spike:   {smoke_alarm_time:%Y-%m-%d %H:%M}")

### ✅ Verify in Fusion

Open *Drum Temperature* in Fusion → **Charts** (or click the inline chart on the detail page). You should see:

- Daily roast cycles — temperature rising to ~220 °C during working hours, dropping back overnight.
- A **flat patch around 10 days ago** at ~28 °C lasting four hours — the drum was off for cleaning.
- A **dramatic, erratic spike to ~260 °C around 5 days ago** — clearly above the normal ~220 ceiling, with chaotic oscillations during the buildup, a sharp crash when the operator kills the drum, and a ~1 h dead zone at ambient while the exhaust duct is cleared.

Open *Bean Moisture* in the same chart — it should show the inverse pattern (~11 % when green, dropping to ~3 % during a roast), with the smoke-alarm anomaly crashing moisture to near ~1 % (scorched beans).

## Step 5 — Activities

`CogniteActivity` represents anything that happens **over a span of time** on an asset — a maintenance work order, an inspection round, a reported incident. It carries scheduled and actual start/end times (inherited from `CogniteSchedulable`) and links to one or more assets through the `assets` direct relation.

Activities are master data on the **CMMS / EAM side** of the house, the systems that own the maintenance lifecycle. They reach CDF through the same [Cognite extractors](https://www.cognite.com/en/product/extractors) that own master data more generally:

| Source | Extractor |
|---|---|
| SAP PM / SAP S/4HANA Asset Management | **Cognite SAP Extractor** — work orders, notifications, operations, confirmations |
| The ODBC-compatible database backing the CMMS | **Cognite DB Extractor** |
| REST-only CMMS APIs without a queryable DB | **CDF Functions** running on a schedule (custom code, not an extractor) |

When activities live in the same graph as the sensor data, a class of question that was previously impossible becomes a single query: *"did the alarm correlate with a known temperature spike?"*, *"how does drift after maintenance compare to drift before?"*, *"which work orders consistently leave equipment under-performing?"* The two anomalies in the temperature trace make sense once we see what happened during those windows.

We write two activities, both on the roasting drum:

1. **Monthly Drum Cleaning** — a scheduled work order that ran ~10 days ago. Uses `scheduled_start_time` / `scheduled_end_time` (planned window from the CMMS). The drum is shut down during cleaning, which is why the temperature was flat at ambient.
2. **Smoke Alarm During Dark Roast** — a reported issue from ~5 days ago. Uses `start_time` / `end_time` (actual incident times). Chaff buildup blocked the exhaust over roughly an hour; the temperature climbed erratically past 260 °C before the alarm fired and an operator killed the drum.

In [ ]:
# Use the same relative timestamps as the datapoints (defined in Step 4)
maintenance = CogniteActivityApply(
    space=SPACE,
    external_id="wo_drum_cleaning",
    name="Roasting Drum — Monthly Cleaning",
    description="Remove chaff buildup, inspect burner nozzles, lubricate drum bearings",
    tags=["maintenance", "scheduled", "monthly"],
    scheduled_start_time=maintenance_start,
    scheduled_end_time=maintenance_end,
    assets=[DirectRelationReference(SPACE, "roasting_drum")],
)

issue = CogniteActivityApply(
    space=SPACE,
    external_id="issue_smoke_alarm",
    name="Smoke Alarm During Dark Roast",
    description=(
        "Smoke detector triggered during French Roast batch #47. "
        "Drum exhaust fan was running at reduced speed. "
        "Temperature climbed erratically past 260 °C over ~1 hour before operator shutdown. "
        "No fire — root cause: chaff buildup in exhaust duct."
    ),
    tags=["issue", "safety", "smoke"],
    start_time=smoke_alarm_time - timedelta(hours=1),
    end_time=smoke_alarm_time + timedelta(minutes=90),
    assets=[DirectRelationReference(SPACE, "roasting_drum")],
)

result = client.data_modeling.instances.apply(nodes=[maintenance, issue])
print(f"Created {len(result.nodes)} activities:")
for n in result.nodes:
    print(f"  {n.space}/{n.external_id}")

### ✅ Verify in Fusion

Search for `Smoke Alarm` in Fusion. You should see:

- The activity *Smoke Alarm During Dark Roast* with status, start/end times, and tags.
- The **Asset** section linking to *Roasting Drum*.
- From *Roasting Drum*'s detail page, the **Activities** tab now lists both the smoke alarm and the monthly cleaning. You can read them alongside the time series chart from the previous step — the timestamps line up with the two anomalies.

## Step 6 — Files

`CogniteFile` represents a document — P&IDs, manuals, inspection reports, datasheets. The **metadata** is a DM node (the same shape as everything else we have written so far); the **content** — the actual bytes — is uploaded separately via `client.files.upload_content(...)`. The two are linked by the file's `instance_id`, so the metadata and the binary stay in sync.

Engineering documents typically live in a content management system, mirrored into CDF by one of the [Cognite extractors](https://www.cognite.com/en/product/extractors):

| Source | Extractor |
|---|---|
| Local file systems / network shares | **Cognite file extractor** |
| OpenText Documentum, Documentum D2 | **Cognite Documentum and D2 Extractor** |
| Other content systems with a REST API (e.g. SharePoint, custom DMS) | **CDF Functions** calling the source's API on a schedule |

We create two files inline so the notebook is self-contained:

1. A **P&ID** PDF — the process diagram, generated with matplotlib — linked to the root asset *Coffee Roastery*.
2. A **maintenance manual** PDF — the standard operating procedure for monthly drum cleaning — linked to *Roasting Drum*.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon, FancyBboxPatch, Circle
from matplotlib.backends.backend_pdf import PdfPages

work_dir = tempfile.mkdtemp(prefix="coffee_roastery_")

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.set_xlim(0, 820)
ax.set_ylim(310, 0)
ax.set_aspect("equal")
ax.axis("off")
fig.patch.set_facecolor("white")

ax.text(
    410,
    25,
    "Coffee Roastery \u2014 Process & Instrumentation Diagram",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="#2c3e50",
)
ax.plot([60, 760], [38, 38], color="#bdc3c7", linewidth=0.5)

# Bean Hopper (trapezoid)
ax.add_patch(
    MplPolygon(
        [(80, 75), (170, 75), (190, 155), (60, 155)], fc="#f5e6cc", ec="#8b6914", lw=2
    )
)
ax.text(
    125,
    108,
    "Bean\nHopper",
    ha="center",
    va="center",
    fontsize=9,
    fontweight="bold",
    color="#2c3e50",
)
ax.text(125, 148, "BH-01", ha="center", fontsize=8, color="#7f8c8d")

# Pipe: Hopper → Drum
ax.annotate(
    "",
    xy=(290, 125),
    xytext=(190, 125),
    arrowprops=dict(arrowstyle="->", color="#34495e", lw=2.5),
)

# Valve V-01
for pts in [[(230, 113), (240, 125), (230, 137)], [(250, 113), (240, 125), (250, 137)]]:
    ax.add_patch(MplPolygon(pts, fc="none", ec="#34495e", lw=1.5))
ax.text(240, 104, "V-01", ha="center", fontsize=7, color="#7f8c8d")

# Roasting Drum
ax.add_patch(
    FancyBboxPatch(
        (300, 72), 187, 106, boxstyle="round,pad=5", fc="#e8c9a0", ec="#8b4513", lw=2
    )
)
ax.text(
    394,
    112,
    "Roasting Drum",
    ha="center",
    fontsize=11,
    fontweight="bold",
    color="#2c3e50",
)
ax.text(394, 137, "RD-01", ha="center", fontsize=9, color="#7f8c8d")

# Gas supply (dashed)
ax.plot([394, 394], [72, 46], color="#e67e22", lw=2, ls="--")
ax.text(394, 42, "Gas Supply", ha="center", fontsize=7, color="#e67e22")

# TI-101 (temperature indicator)
ax.plot([360, 360], [178, 205], color="#e74c3c", lw=1.5)
ax.add_patch(Circle((360, 222), 18, fc="white", ec="#e74c3c", lw=2))
ax.text(
    360,
    222,
    "TI",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="#e74c3c",
)
ax.text(360, 250, "TI-101", ha="center", fontsize=8, color="#7f8c8d")

# MI-101 (moisture indicator)
ax.plot([428, 428], [178, 205], color="#3498db", lw=1.5)
ax.add_patch(Circle((428, 222), 18, fc="white", ec="#3498db", lw=2))
ax.text(
    428,
    222,
    "MI",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="#3498db",
)
ax.text(428, 250, "MI-101", ha="center", fontsize=8, color="#7f8c8d")

# Pipe: Drum → Cooling Tray
ax.annotate(
    "",
    xy=(590, 125),
    xytext=(491, 125),
    arrowprops=dict(arrowstyle="->", color="#34495e", lw=2.5),
)

# Cooling Tray
ax.add_patch(
    FancyBboxPatch(
        (592, 90), 164, 70, boxstyle="round,pad=3", fc="#d5e8f0", ec="#2980b9", lw=2
    )
)
ax.text(
    674,
    120,
    "Cooling Tray",
    ha="center",
    fontsize=11,
    fontweight="bold",
    color="#2c3e50",
)
ax.text(674, 143, "CT-01", ha="center", fontsize=9, color="#7f8c8d")

# Legend
ax.add_patch(
    FancyBboxPatch(
        (55, 275), 710, 26, boxstyle="round,pad=2", fc="#f8f9fa", ec="#ecf0f1"
    )
)
ax.text(
    410,
    288,
    "TI = Temperature Indicator  |  MI = Moisture Indicator"
    "  |  V = Manual Valve  |  ---- = Gas Line",
    ha="center",
    fontsize=7,
    color="#95a5a6",
)

pid_path = Path(work_dir) / "pid_coffee_roastery.pdf"
fig.savefig(str(pid_path), format="pdf", bbox_inches="tight", facecolor="white")

plt.show()
print(f"  {pid_path.name} ({pid_path.stat().st_size:,} bytes)")

# \u2550\u2550\u2550 Maintenance Manual (SOP) as PDF \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550

sop_pages = [
    [
        ("title", "STANDARD OPERATING PROCEDURE"),
        ("subtitle", "SOP-RD-001  |  Rev. 2  |  Effective: 2026-01-15"),
        ("subtitle", "Roasting Drum \u2014 Monthly Cleaning & Inspection"),
        ("subtitle", "Equipment: RD-01 (Gas-Fired Rotating Drum, 15 kg batch)"),
        ("blank", ""),
        ("heading", "1. PURPOSE"),
        ("body", "Remove chaff buildup, inspect critical components, and"),
        ("body", "ensure safe and consistent roast quality."),
        ("blank", ""),
        ("heading", "2. SAFETY REQUIREMENTS"),
        ("body", "\u25a1  Lock out gas supply (close V-01, tag out main gas valve)"),
        ("body", "\u25a1  Allow drum to cool below 50 \u00b0C (confirm on TI-101)"),
        ("body", "\u25a1  Wear heat-resistant gloves and safety glasses"),
        ("body", "\u25a1  Ensure fire extinguisher is accessible"),
        ("blank", ""),
        ("heading", "3. PROCEDURE"),
        ("blank", ""),
        ("subheading", "3.1  Preparation"),
        ("body", "  \u2022 Complete current roast batch and run cooling cycle"),
        ("body", "  \u2022 Shut down drum motor and gas supply"),
        ("body", "  \u2022 Confirm drum temperature < 50 \u00b0C on TI-101"),
        ("blank", ""),
        ("subheading", "3.2  Chaff Removal"),
        ("body", "  \u2022 Open drum access panel"),
        ("body", "  \u2022 Vacuum chaff from drum interior, exhaust duct, and cyclone"),
        ("body", "  \u2022 Inspect exhaust duct for blockages"),
        ("body", "    (ref: incident issue_smoke_alarm \u2014 chaff buildup"),
        ("body", "     caused smoke detector activation)"),
        ("blank", ""),
        ("subheading", "3.3  Burner Inspection"),
        ("body", "  \u2022 Remove burner cover and inspect nozzle tips"),
        ("body", "  \u2022 Clean nozzles with brass brush"),
        ("body", "  \u2022 Check gas line connections for leaks (soap-bubble test)"),
    ],
    [
        ("subheading", "3.4  Drum Bearings"),
        ("body", "  \u2022 Apply food-grade lubricant to drum shaft bearings"),
        ("body", "  \u2022 Rotate drum by hand \u2014 verify smooth operation"),
        ("body", "  \u2022 Check belt tension on drive motor"),
        ("blank", ""),
        ("subheading", "3.5  Reassembly & Test"),
        ("body", "  \u2022 Close access panel and restore gas supply (open V-01)"),
        ("body", "  \u2022 Run empty drum at 150 \u00b0C for 10 minutes"),
        ("body", "  \u2022 Verify TI-101 reads within +/- 5 \u00b0C of setpoint"),
        ("body", "  \u2022 Check MI-101 reads ambient moisture level"),
        ("blank", ""),
        ("blank", ""),
        ("heading", "4. SIGN-OFF"),
        ("body", ""),
        ("body", "  Performed by: _________________________  Date: ____________"),
        ("body", ""),
        ("body", "  Reviewed by:  _________________________  Date: ____________"),
    ],
]

manual_path = Path(work_dir) / "sop_drum_cleaning.pdf"

with PdfPages(str(manual_path)) as pdf:
    for page_num, page_lines in enumerate(sop_pages, 1):
        fig_sop = plt.figure(figsize=(8.5, 11))
        fig_sop.patch.set_facecolor("white")

        # Header
        fig_sop.text(
            0.5,
            0.97,
            "COFFEE ROASTERY \u2014 STANDARD OPERATING PROCEDURE",
            ha="center",
            fontsize=8,
            color="#777",
        )
        fig_sop.text(
            0.08, 0.958, "\u2500" * 100, fontsize=4, color="#ccc", family="monospace"
        )

        y = 0.93
        for style, text in page_lines:
            if style == "title":
                fig_sop.text(0.5, y, text, ha="center", fontsize=16, fontweight="bold")
                y -= 0.04
            elif style == "subtitle":
                fig_sop.text(0.5, y, text, ha="center", fontsize=10, color="#555")
                y -= 0.028
            elif style == "heading":
                y -= 0.008
                fig_sop.text(0.08, y, text, fontsize=11, fontweight="bold")
                y -= 0.03
            elif style == "subheading":
                fig_sop.text(0.1, y, text, fontsize=10, fontweight="bold", color="#333")
                y -= 0.026
            elif style == "body":
                fig_sop.text(0.1, y, text, fontsize=9, family="monospace")
                y -= 0.022
            elif style == "blank":
                y -= 0.015

        # Footer
        fig_sop.text(0.08, 0.02, "SOP-RD-001 Rev. 2", fontsize=7, color="#999")
        fig_sop.text(
            0.5,
            0.02,
            f"Page {page_num} of {len(sop_pages)}",
            ha="center",
            fontsize=7,
            color="#999",
        )

        pdf.savefig(fig_sop)
        plt.close(fig_sop)

print(f"  {manual_path.name} ({manual_path.stat().st_size:,} bytes)")

In [ ]:
pid_file = CogniteFileApply(
    space=SPACE,
    external_id="pid_coffee_roastery",
    name="P&ID — Coffee Roastery",
    description="Process & Instrumentation Diagram for the coffee roastery",
    mime_type="application/pdf",
    assets=[DirectRelationReference(SPACE, "coffee_roastery")],
)

manual_file = CogniteFileApply(
    space=SPACE,
    external_id="sop_drum_cleaning",
    name="SOP — Roasting Drum Monthly Cleaning",
    description="Standard operating procedure for roasting drum maintenance",
    mime_type="application/pdf",
    assets=[DirectRelationReference(SPACE, "roasting_drum")],
)

result = client.data_modeling.instances.apply(nodes=[pid_file, manual_file])
print(f"Created {len(result.nodes)} file nodes:")
for n in result.nodes:
    print(f"  {n.space}/{n.external_id}")

print("\nWaiting a few seconds for CDF to sync the file nodes...")
time.sleep(2)

client.files.upload_content(
    str(pid_path), instance_id=NodeId(SPACE, "pid_coffee_roastery")
)
print(f"Uploaded P&ID PDF ({pid_path.stat().st_size:,} bytes)")

client.files.upload_content(
    str(manual_path), instance_id=NodeId(SPACE, "sop_drum_cleaning")
)
print(f"Uploaded maintenance manual ({manual_path.stat().st_size:,} bytes)")

### ✅ Verify in Fusion

Search for `P&ID` in Fusion. You should see:

- The file *P&ID — Coffee Roastery*. Click it — Fusion previews the PDF inline. The diagram is the same one matplotlib generated above.
- The **Asset** section links to *Coffee Roastery*; clicking through brings you to the root asset, which now has this file under **Files**.

Search for `SOP` to find the maintenance manual, linked to *Roasting Drum*.

## Step 7 — P&ID Annotations

A P&ID file on its own is just a picture. The value comes from **linking regions of the drawing to the asset graph** — clicking a tag in the P&ID jumps straight to live sensor data and maintenance history.

CDF represents these links as **`CogniteDiagramAnnotation` edges**. An *edge* is different from the *direct relations* we have used so far: it's a separate, addressable instance with its own properties — start node, end node, type, and in this case a bounding box, the detected text, a confidence score, and a status (`Suggested` / `Approved` / `Rejected`). Edges are the right tool when the connection itself carries data; direct relations are the right tool when it's a plain pointer.

In a real pipeline, annotations are produced by **CDF Diagram Detection** (`client.diagrams.detect()`) — an OCR + ML service that scans the drawing and matches detected text against a list of entities you supply. Engineers then review the suggestions in **Fusion Canvas** and approve or reject each one. New revisions of a P&ID re-run detection automatically when the *contextualization deployment pack* is configured.

We do the same flow in four sub-steps:

1. **7a** — write P&ID reference tags onto the assets and time series. The tags become searchable text — what Diagram Detection will look for.
2. **7b** — read the entities back from CDF and shape them as input for `diagrams.detect()`.
3. **7c** — call `diagrams.detect()` on the P&ID file we uploaded above.
4. **7d** — turn each detection result into a `CogniteDiagramAnnotation` edge.

### 7a — Add P&ID reference tags

Each piece of equipment in a P&ID has a **tag number** (e.g. `RD-01` for the Roasting Drum,
`TI-101` for the temperature indicator). We store these as `tags` on the corresponding
`CogniteAsset` and `CogniteTimeSeries` nodes.

This is an **upsert** — we only set the `tags` property; all other properties remain unchanged.

In [ ]:
describable_view = ViewId(CDM_SPACE, "CogniteDescribable", CDM_VERSION)

pid_tag_mapping = {
    "bean_hopper": ["BH-01"],
    "roasting_drum": ["RD-01"],
    "cooling_tray": ["CT-01"],
    "drum_temperature": ["TI-101"],
    "bean_moisture": ["MI-101"],
}

tag_updates = [
    NodeApply(
        space=SPACE,
        external_id=ext_id,
        sources=[NodeOrEdgeData(source=describable_view, properties={"tags": tags})],
    )
    for ext_id, tags in pid_tag_mapping.items()
]

result = client.data_modeling.instances.apply(nodes=tag_updates)
print(f"Added P&ID reference tags to {len(result.nodes)} nodes:")
for ext_id, tags in pid_tag_mapping.items():
    print(f"  {ext_id} → {tags}")

### 7b — Build detection entities from the API

We fetch all assets and time series from the space and build the `entities` list
that `diagrams.detect()` needs. Each entity includes a `search_tag` — the P&ID tag
that the detection service will look for in the drawing.

In [ ]:
asset_view = ViewId(CDM_SPACE, "CogniteAsset", CDM_VERSION)
ts_view = ViewId(CDM_SPACE, "CogniteTimeSeries", CDM_VERSION)
ANNOTATION_VIEW = ViewId(CDM_SPACE, "CogniteDiagramAnnotation", CDM_VERSION)

assets = client.data_modeling.instances.list(
    space=SPACE, sources=[asset_view], limit=None
)
timeseries = client.data_modeling.instances.list(
    space=SPACE, sources=[ts_view], limit=None
)

print(f"Fetched {len(assets)} assets and {len(timeseries)} time series from '{SPACE}'")

# Build the entities list: one entry per node that has a P&ID tag.
# `search_tag` is the string Diagram Detection looks for in the drawing.
entities = []
for node in list(assets) + list(timeseries):
    tags = (node["tags"] or []) if "tags" in node else []
    if not tags:
        continue
    entities.append(
        {
            "external_id": node.external_id,
            "space": node.space,
            "annotation_type_external_id": "diagrams.AssetLink",
            "name": node["name"],
            "search_tag": tags[0],
        }
    )

print(f"\nBuilt {len(entities)} entities for detection:")
for e in entities:
    print(f"  {e['name']:20s}  search_tag={e['search_tag']}")

### 7c — Run diagram detection

This can take some time depending on cluster load etc.

In [ ]:
detect_job = client.diagrams.detect(
    file_instance_ids=[NodeId(SPACE, "pid_coffee_roastery")],
    entities=entities,
    search_field="search_tag",
    min_tokens=1,
)
detect_job.result

In [ ]:
print(
    f"Detection complete — {len(detect_job.result.get('items', []))} file(s) processed"
)
for item in detect_job.result.get("items", []):
    file_ext_id = item.get("fileInstanceId", {}).get("externalId", "?")
    annotations = item.get("annotations", [])
    print(f"  {file_ext_id}: {len(annotations)} annotation(s)")
    for ann in annotations[:10]:
        entity_names = [
            e.get("name", e.get("search_tag", "?")) for e in ann.get("entities", [])
        ]
        print(
            f"    text='{ann.get('text', '?')}'  conf={ann.get('confidence', 0):.2f}  → {entity_names}"
        )

### 7d — Create annotation edges

> **Idempotency note:** we use `uuid.uuid4()` as the edge `external_id`, so each re-run creates a *new* set of annotation edges rather than upserting the existing ones. That keeps the demo simple but means you should run the cleanup cell before re-running this section. In production, derive a deterministic `external_id` from the source/target/page/box so re-runs upsert cleanly.

In [ ]:
CONFIDENCE_THRESHOLD = 0.5

edge_applies = []
for item in detect_job.result.get("items", []):
    file_id_dict = item.get("fileInstanceId", {})
    file_space = file_id_dict.get("space", SPACE)
    file_ext_id = file_id_dict.get("externalId", "")

    for ann in item.get("annotations", []):
        if ann.get("confidence", 0) < CONFIDENCE_THRESHOLD:
            continue
        for entity in ann.get("entities", []):
            edge_applies.append(
                EdgeApply(
                    space=SPACE,
                    external_id=str(uuid.uuid4()),
                    type=DirectRelationReference(
                        CDM_SPACE, entity["annotation_type_external_id"]
                    ),
                    start_node=DirectRelationReference(file_space, file_ext_id),
                    end_node=DirectRelationReference(
                        entity["space"], entity["external_id"]
                    ),
                    sources=[
                        NodeOrEdgeData(
                            source=ANNOTATION_VIEW,
                            properties={
                                "confidence": ann["confidence"],
                                "status": "Suggested",
                                "startNodePageNumber": ann["region"]["page"],
                                "startNodeXMin": min(
                                    v["x"] for v in ann["region"]["vertices"]
                                ),
                                "startNodeYMin": min(
                                    v["y"] for v in ann["region"]["vertices"]
                                ),
                                "startNodeXMax": max(
                                    v["x"] for v in ann["region"]["vertices"]
                                ),
                                "startNodeYMax": max(
                                    v["y"] for v in ann["region"]["vertices"]
                                ),
                                "startNodeText": ann["text"],
                            },
                        )
                    ],
                )
            )

print(f"Found {len(edge_applies)} annotations above {CONFIDENCE_THRESHOLD} confidence")

if edge_applies:
    result = client.data_modeling.instances.apply(edges=edge_applies)
    print(f"Created {len(result.edges)} annotation edges (P&ID → entities):")
    for e in result.edges:
        print(f"  {e.space}/{e.external_id[:8]}…")
else:
    print("No annotations met the confidence threshold.")

### ✅ Verify in Fusion

Open *P&ID — Coffee Roastery* in Fusion. You should see:

- Coloured rectangles overlaid on each tag in the drawing — `BH-01`, `RD-01`, `CT-01`, `TI-101`, `MI-101`. These are the bounding boxes from the annotations we just wrote.
- Hovering over a rectangle shows the linked entity (asset or time series); clicking jumps to that entity's detail page. The graph is now navigable straight from the diagram.
- On *Bean Hopper* (or any equipment asset), the **P&IDs** panel now references this file — the link is bidirectional, even though we only wrote it once.

## Cleanup

Removes only what **this notebook** created — the four assets, two time series (with datapoints), two activities, two files, and any annotation edges that point at those nodes.

If you've also run the data-model-extension or 3D notebook into the same space, those are left alone (each has its own cleanup cell).

Set `run_cleanup = True` to execute.

In [ ]:
run_cleanup = True

OWNED_NODES = [
    # assets
    "coffee_roastery",
    "bean_hopper",
    "roasting_drum",
    "cooling_tray",
    # time series
    "drum_temperature",
    "bean_moisture",
    # activities
    "wo_drum_cleaning",
    "issue_smoke_alarm",
    # files
    "pid_coffee_roastery",
    "sop_drum_cleaning",
]

if run_cleanup:
    owned_node_ids = [NodeId(SPACE, ext_id) for ext_id in OWNED_NODES]

    # Annotation edges whose start_node is one of our files — uniquely identify
    # them by the start_node + view, then delete by id.
    file_node_ids = [
        NodeId(SPACE, "pid_coffee_roastery"),
        NodeId(SPACE, "sop_drum_cleaning"),
    ]
    ann_view = ViewId(CDM_SPACE, "CogniteDiagramAnnotation", "v1")
    annotation_edges = client.data_modeling.instances.list(
        instance_type="edge",
        sources=[ann_view],
        filter=filters.In(
            ["edge", "startNode"],
            [{"space": n.space, "externalId": n.external_id} for n in file_node_ids],
        ),
        limit=None,
    )

    if annotation_edges:
        client.data_modeling.instances.delete(edges=annotation_edges.as_ids())
        print(f"Deleted {len(annotation_edges)} annotation edge(s)")

    # Datapoints first (so nothing dangles), then the time series nodes themselves.
    for ts_ext in ("drum_temperature", "bean_moisture"):
        client.time_series.data.delete_range(
            start=0,
            end="now",
            instance_id=NodeId(SPACE, ts_ext),
        )
    print("Deleted datapoints for both time series")

    client.data_modeling.instances.delete(nodes=owned_node_ids)
    print(f"Deleted {len(owned_node_ids)} node(s)")
    print("Cleanup complete.")
else:
    print("Cleanup skipped — set run_cleanup = True to execute.")